# Laya Model Training for WCAG 2.2 Violation Detection

Fine-tunes the Laya model to detect WCAG 2.2 accessibility violations.
Output: ONNX model compatible with layaForWeb.

In [ ]:
!pip install -q torch transformers datasets onnx onnxruntime laya

## Load WCAG 2.2 Training Data

Option A: Use existing WCAG violation datasets
Option B: Generate synthetic data using LLM
Option C: Combine multiple sources

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HFDataset

# Define WCAG 2.2 criteria
WCAG_22_CRITERIA = [
    {"criterion": "1.1.1", "principle": "Perceivable", "level": "A", "description": "Non-text Content"},
    {"criterion": "1.2.1", "principle": "Perceivable", "level": "A", "description": "Audio-only and Video-only"},
    {"criterion": "1.2.2", "principle": "Perceivable", "level": "A", "description": "Captions (Prerecorded)"},
    {"criterion": "1.3.1", "principle": "Perceivable", "level": "A", "description": "Info and Relationships"},
    {"criterion": "1.4.1", "principle": "Perceivable", "level": "A", "description": "Use of Color"},
]
print(f"Loaded {len(WCAG_22_CRITERIA)} WCAG criteria")


In [ ]:
def generate_synthetic_training_data(num_samples=1000):
    """Generate synthetic WCAG violation training data."""
    training_data = []
    
    for i in range(num_samples):
        text = f"Web page {i}: Review for WCAG 2.2 accessibility compliance."
        # Randomly assign violations
        violations = []
        for criterion in WCAG_22_CRITERIA:
            if i % (hash(criterion['criterion']) % 7 + 1) == 0:
                violations.append(criterion)
        
        training_data.append({
            "text": text,
            "violations": violations
        })
    
    return training_data

training_data = generate_synthetic_training_data(500)
print(f"Generated {len(training_data)} synthetic training samples")

In [ ]:
from datasets import Dataset

def create_hf_dataset(data):
    """Convert list of dicts to HuggingFace Dataset."""
    texts = [item["text"] for item in data]
    violation_lists = [item["violations"] for item in data]
    
    return Dataset.from_dict({
        "text": texts,
        "violations": violation_lists
    })

hf_dataset = create_hf_dataset(training_data)
train_test_split = hf_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
val_dataset = train_test_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
## Load Base Laya Model

from transformers import AutoTokenizer, AutoModel

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained("convaiinnovations/laya")
base_model = AutoModel.from_pretrained("convaiinnovations/laya")
base_model.to(device)

print(f"Base model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
def tokenize_function(examples):
    """Tokenize text inputs for the model."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("Tokenization complete")

In [ ]:
## Multi-Label Classification Head

class LayaWCAGModel(nn.Module):
    def __init__(self, base_model, num_criteria):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_criteria)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # CLS token
        logits = self.classifier(pooled)
        return logits

num_criteria = len(WCAG_22_CRITERIA)
model = LayaWCAGModel(base_model, num_criteria)
model.to(device)

print(f"Model initialized with {num_criteria} criteria classification head")

In [ ]:
## Training Configuration

# Training hyperparameters
learning_rate = 2e-5
batch_size = 8
num_epochs = 3
num_criteria = len(WCAG_22_CRITERIA)

# Create data collator
from torch.utils.data import DataLoader

# Simple training arguments
print(f"Training config:")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {num_epochs}")
print(f"  Criteria: {num_criteria}")

In [ ]:
## DataLoader with Label Encoding

def create_label_map(criterias):
    """Create a mapping from criterion IDs to indices."""
    return {c['criterion']: i for i, c in enumerate(criterias)}

label_map = create_label_map(WCAG_22_CRITERIA)

def encode_labels(examples):
    """Convert violations list to multi-hot encoded labels."""
    labels = []
    for violation_list in examples['violations']:
        label = [0] * num_criteria
        for v in violation_list:
            idx = label_map.get(v['criterion'])
            if idx is not None:
                label[idx] = 1
        labels.append(label)
    return {'labels': labels}

tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_loader = DataLoader(tokenized_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(tokenized_val, batch_size=batch_size, shuffle=False)

print(f"Created {len(train_loader)} training batches, {len(val_loader)} validation batches")

In [ ]:
## Training Loop

import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Optimizer
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

criterion = nn.BCEWithLogitsLoss()

print("Model ready for training")
print(f"Total training steps: {total_steps}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")